In [1]:
import warnings
import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
veri = pd.read_csv("C:/Users/kerem/Desktop/Akıllı Trafik/Modeller/proje.csv")
veri=veri.copy()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda  x: '%.3f' % x)
pd.set_option('display.width', 1000)
warnings.filterwarnings('ignore')

In [3]:
# ---------------------------
# USER ADJUSTABLE HYPERPARAMS
# ---------------------------
HIST = 35          # geçmiş penceresi (periyot cinsinden; eğer 1 gün=5 periyot ise 7 gün = 35)
HORIZON = 12       # tahmin adımı (periyot cinsinden)
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-3
TEST_RATIO = 0.2
VAL_RATIO = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "DL_MULTI_FEATURE_RESULTS"
os.makedirs(OUT_DIR, exist_ok=True)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", DEVICE)

Device: cuda


In [4]:
# ---------------------------
# 1) PREPROCESS: ensure columns & period_code one-hot
# ---------------------------
veri = veri.copy()
# required
if 'DATE' not in veri.columns or 'GEOHASH_SHORT' not in veri.columns or 'DENSITY' not in veri.columns:
    raise ValueError("veri must contain 'DATE', 'GEOHASH_SHORT', and 'DENSITY' columns.")

# ensure datetime
veri['DATE'] = pd.to_datetime(veri['DATE'], errors='coerce')
veri = veri.dropna(subset=['DATE','GEOHASH_SHORT','DENSITY']).reset_index(drop=True)

# If period_code provided as 0-4 integers, use; otherwise, attempt to create from TIME_PERIOD column
if 'period_code' not in veri.columns:
    if 'TIME_PERIOD' in veri.columns:
        # if already string labels convert to categorical codes
        if veri['TIME_PERIOD'].dtype == 'O':
            time_map = {
                'Night': 0,
                'Morning_Peak': 1,
                'Daytime': 2,
                'Evening_Peak': 3,
                'Late_Evening': 4
            }
            # Eğer veri setinizde küçük/büyük harf farkı varsa .str.strip() vb. kullanın
            veri['period_code'] = veri['TIME_PERIOD'].map(time_map).fillna(0).astype(int)
        else:
            veri['period_code'] = veri['TIME_PERIOD'].astype(int)
    else:
        # fallback: try to derive from hour if available
        if 'hour' in veri.columns:
            # map hour to 0-4 buckets (custom; user may adapt)
            bins = [0,6,10,15,19,24]  # example: night(0-6), morning(6-10), noon(10-15), afternoon(15-19), night(19-24)
            labels = [4,0,1,2,3]      # arbitrary mapping -> you should adjust to your definitions
            veri['period_code'] = pd.cut(veri['hour'], bins=bins, labels=labels, include_lowest=True).astype(float).fillna(0).astype(int)
        else:
            # if nothing exists, set to 0
            veri['period_code'] = 0
            print("Warning: period_code missing and couldn't derive — set to 0")

# Optional features: build list of features to include (first must be DENSITY)
base_features = ['DENSITY']
optional_candidates = ['AVERAGE_SPEED','NUMBER_OF_VEHICLES','rain_encoded','temp']
features_present = [f for f in base_features + optional_candidates if f in veri.columns]

# We'll encode period_code as one-hot channels
# Create one-hot columns in the pivot step (not in dataframe as separate columns per geohash)
N_PERIODS = max(veri['period_code'].max(), 0) + 1
if N_PERIODS < 2: N_PERIODS = 5  # assume 5 if odd

print("Detected feature columns:", features_present)
print("Detected period_code unique values:", sorted(veri['period_code'].unique()))

Detected feature columns: ['DENSITY', 'AVERAGE_SPEED', 'NUMBER_OF_VEHICLES']
Detected period_code unique values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [5]:
# ---------------------------
# 2) PIVOT FEATURES -> (T, N, C)
# ---------------------------
# We'll create pivot for each numeric feature: pivot(index=(DATE,period_code), columns=GEOHASH_SHORT)
def pivot_feature_matrix(veri, node_list, feature):
    piv = veri.pivot_table(index=['DATE','period_code'], columns='GEOHASH_SHORT', values=feature)
    piv = piv.reindex(columns=node_list)
    piv = piv.sort_index().fillna(method='ffill').fillna(method='bfill')
    # if still NaN, fill with 0
    piv = piv.fillna(0.0)
    return piv

# Node list order
node_list = list(veri['GEOHASH_SHORT'].drop_duplicates().tolist())
print("Num nodes:", len(node_list))

# build pivot for density + other features
mats = []
index_ref = None

# 1) density (channel 0)
p_density = pivot_feature_matrix(veri, node_list, 'DENSITY')
index_ref = p_density.index
mats.append(p_density.values.astype(np.float32))

# 2) optional numeric features
for feat in optional_candidates:
    if feat in veri.columns:
        piv = pivot_feature_matrix(veri, node_list, feat)
        # align index
        piv = piv.reindex(index_ref).fillna(method='ffill').fillna(method='bfill').fillna(0.0)
        mats.append(piv.values.astype(np.float32))

# 3) period_code -> one-hot expanded per period
# Build per (DATE, period_code) a one-hot vector, then repeat for nodes
period_idx = index_ref.get_level_values(1).astype(int).values  # period_code values aligned with index_ref
# For each period value create matrix (T, N)
period_onehots = []
for p in range(N_PERIODS):
    vec = (period_idx == p).astype(np.float32)  # (T,)
    mat = np.tile(vec.reshape(-1,1), (1, len(node_list)))  # (T, N)
    period_onehots.append(mat)
# append each period one-hot as separate channel
for mat in period_onehots:
    mats.append(mat.astype(np.float32))

# Stack along channel axis -> (T, N, C)
data = np.stack(mats, axis=-1)
T, N, C = data.shape
print("Final tensor shape (T, N, C):", data.shape)

Num nodes: 199
Final tensor shape (T, N, C): (1368, 199, 8)


In [6]:
# ---------------------------
# 3) SCALING: scale each channel separately across all T,N (fit per-channel scaler)
# ---------------------------
scalers = []
data_scaled = np.zeros_like(data, dtype=np.float32)
for c in range(C):
    arr = data[:,:,c].reshape(-1,1)
    sc = MinMaxScaler(feature_range=(0,1))
    sc.fit(arr)
    arr_s = sc.transform(arr).reshape(T,N)
    data_scaled[:,:,c] = arr_s
    scalers.append(sc)
print("Scaling completed.")

Scaling completed.


In [7]:
# ---------------------------
# 4) CREATE SEQUENCES (X: S,HIST,N,C) (Y: S,HORIZON,N) target = density channel (index 0)
# ---------------------------
def create_sequences_multi(data_arr, hist, horizon, stride=1):
    # data_arr: (T,N,C)
    T = data_arr.shape[0]
    X_lst, Y_lst = [], []
    for t in range(0, T - hist - horizon + 1, stride):
        x = data_arr[t:t+hist]               # (hist, N, C)
        y = data_arr[t+hist:t+hist+horizon, :, 0]  # density channel -> (horizon, N)
        X_lst.append(x)
        Y_lst.append(y)
    X = np.stack(X_lst).astype(np.float32) if len(X_lst)>0 else np.zeros((0,hist,N,C), dtype=np.float32)
    Y = np.stack(Y_lst).astype(np.float32) if len(Y_lst)>0 else np.zeros((0,horizon,N), dtype=np.float32)
    return X, Y

X, Y = create_sequences_multi(data_scaled, HIST, HORIZON, stride=1)
S = len(X)
print("Num samples:", S, "X shape:", X.shape, "Y shape:", Y.shape)
if S == 0:
    raise ValueError("No sequences created. Reduce HIST/HORIZON or check data length.")

Num samples: 1322 X shape: (1322, 35, 199, 8) Y shape: (1322, 12, 199)


In [8]:
# ---------------------------
# 5) SPLIT: time-based
# ---------------------------
test_size = int(TEST_RATIO * S)
val_size = int(VAL_RATIO * (S - test_size))
train_size = S - test_size - val_size

X_train = X[:train_size]; Y_train = Y[:train_size]
X_val   = X[train_size:train_size+val_size]; Y_val = Y[train_size:train_size+val_size]
X_test  = X[train_size+val_size:]; Y_test = Y[train_size+val_size:]

print("Split -> Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))

Split -> Train: 953 Val: 105 Test: 264


In [9]:
# ---------------------------
# 6) DataLoader
# ---------------------------
class SeqMultiDS(Dataset):
    def __init__(self, X, Y):
        # X: (S, hist, N, C), Y: (S, horizon, N)
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

train_loader = DataLoader(SeqMultiDS(X_train, Y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SeqMultiDS(X_val, Y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(SeqMultiDS(X_test, Y_test), batch_size=BATCH_SIZE, shuffle=False)

In [10]:
# ---------------------------
# 7) MODELS (adapted to multi-feature)
# Input X: (B, HIST, N, C)
# We'll flatten node*channel per time-step: size per time = N*C
# LSTM/CNN-LSTM/Transformer operate on sequence of vectors of size N*C
# ---------------------------
INPUT_DIM = N * C

# --- LSTM ---
class ModelLSTM_Multi(nn.Module):
    def __init__(self, hist, N, C, hidden=256, layers=2, horizon=HORIZON):
        super().__init__()
        self.input_dim = N * C
        self.lstm = nn.LSTM(input_size=self.input_dim, hidden_size=hidden, num_layers=layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden, horizon * N)
        self.N = N
        self.horizon = horizon
    def forward(self, x):
        # x: (B, hist, N, C)
        B = x.size(0)
        x = x.view(B, x.size(1), -1)  # (B, hist, N*C)
        out, _ = self.lstm(x)
        last = out[:, -1, :]          # (B, hidden)
        y = self.fc(last).view(B, self.horizon, self.N)
        return y

In [11]:
# --- CNN-LSTM ---
class ModelCNNLSTM_Multi(nn.Module):
    def __init__(self, hist, N, C, cnn_ch=128, lstm_hidden=256, horizon=HORIZON):
        super().__init__()
        self.N = N; self.C = C
        self.input_dim = N * C
        # we'll treat input as (B, input_dim, hist) for Conv1d
        self.conv = nn.Conv1d(in_channels=self.input_dim, out_channels=cnn_ch, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm1d(cnn_ch)
        self.lstm = nn.LSTM(input_size=cnn_ch, hidden_size=lstm_hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(lstm_hidden, horizon * N)
        self.horizon = horizon
    def forward(self, x):
        B = x.size(0)
        # x -> (B, hist, N, C) -> (B, N*C, hist)
        xt = x.permute(0,1,2,3).contiguous().view(B, x.size(1), -1)  # (B, hist, N*C)
        xt = xt.permute(0,2,1)  # (B, N*C, hist)
        conv = torch.relu(self.bn(self.conv(xt)))  # (B, cnn_ch, hist)
        conv = conv.permute(0,2,1)  # (B, hist, cnn_ch)
        out, _ = self.lstm(conv)
        last = out[:, -1, :]       # (B, lstm_hidden)
        y = self.fc(last).view(B, self.horizon, self.N)
        return y

In [12]:
# --- Transformer ---
class ModelTransformer_Multi(nn.Module):
    def __init__(self, hist, N, C, d_model=256, heads=8, layers=2, ff=512, horizon=HORIZON):
        super().__init__()
        self.input_dim = N * C
        self.proj = nn.Linear(self.input_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=heads, dim_feedforward=ff, batch_first=True)
        self.tf = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.fc = nn.Linear(d_model, horizon * N)
        self.N = N; self.horizon = horizon
    def forward(self, x):
        B = x.size(0)
        x = x.view(B, x.size(1), -1)   # (B, hist, N*C)
        x = self.proj(x)               # (B, hist, d_model)
        out = self.tf(x)               # (B, hist, d_model)
        last = out[:, -1, :]           # (B, d_model)
        y = self.fc(last).view(B, self.horizon, self.N)
        return y

In [13]:
# ---------------------------
# 8) TRAIN / EVAL UTILITIES
# ---------------------------
def train_one_epoch(model, loader, opt, criterion):
    model.train()
    total_loss = 0.0
    cnt = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE); yb = yb.to(DEVICE)
        opt.zero_grad()
        preds = model(xb)            # (B, horizon, N)
        loss = criterion(preds, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item() * xb.size(0)
        cnt += xb.size(0)
    return total_loss / cnt

def evaluate_model(model, loader, criterion):
    model.eval()
    losses = []
    preds_all, trues_all = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            preds = model(xb)
            loss = criterion(preds, yb)
            losses.append(loss.item())
            preds_all.append(preds.cpu().numpy())
            trues_all.append(yb.cpu().numpy())
    if len(preds_all)==0:
        return np.nan, None, None
    preds = np.concatenate(preds_all, axis=0)
    trues = np.concatenate(trues_all, axis=0)
    return np.mean(losses), preds, trues

def inverse_scale_preds(preds_scaled, scalers):
    # preds_scaled: (S, H, N) scaled by channel 0's scaler applied previously (density channel scaler)
    # Our density scaler is scalers[0] fit to flattened density values
    S, H, N = preds_scaled.shape
    arr = preds_scaled.reshape(-1,1)
    inv = scalers[0].inverse_transform(arr).reshape(S, H, N)
    return inv

def compute_metrics(preds_inv, trues_inv):
    preds_flat = preds_inv.reshape(-1)
    trues_flat = trues_inv.reshape(-1)
    mae = mean_absolute_error(trues_flat, preds_flat)
    rmse = np.sqrt(mean_squared_error(trues_flat, preds_flat))
    r2 = r2_score(trues_flat, preds_flat)
    return mae, rmse, r2

In [14]:
# ---------------------------
# 9) RUN: train each model
# ---------------------------
models = {
    "LSTM": ModelLSTM_Multi(HIST, N, C, hidden=256, layers=2, horizon=HORIZON),
    "CNNLSTM": ModelCNNLSTM_Multi(HIST, N, C, cnn_ch=128, lstm_hidden=256, horizon=HORIZON),
    "Transformer": ModelTransformer_Multi(HIST, N, C, d_model=256, heads=8, layers=2, ff=512, horizon=HORIZON)
}

results = {}
criterion = nn.MSELoss()

for name, model in models.items():
    print("\n=== TRAINING", name, "===")
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-6)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    best_val_loss = np.inf
    best_state = None
    patience_cnt = 0

    for ep in range(1, EPOCHS+1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, opt, criterion)
        val_loss, val_preds_s, val_trues_s = evaluate_model(model, val_loader, criterion)
        scheduler.step(val_loss if not np.isnan(val_loss) else train_loss)
        t1 = time.time()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            torch.save(best_state, os.path.join(OUT_DIR, f"{name}_best.pth"))
            patience_cnt = 0
        else:
            patience_cnt += 1

        if ep % 2 == 0 or ep==1:
            print(f"{name} Epoch {ep}/{EPOCHS} | TrainLoss: {train_loss:.6f} | ValLoss: {val_loss:.6f} | time {t1-t0:.1f}s")

        if patience_cnt > 6:
            print("Early stopping triggered.")
            break

    # load best
    model.load_state_dict(torch.load(os.path.join(OUT_DIR, f"{name}_best.pth")))
    # eval on test
    test_loss, test_preds_s, test_trues_s = evaluate_model(model, test_loader, criterion)
    # inverse scale predictions (density channel only)
    test_preds_inv = inverse_scale_preds(test_preds_s, scalers)
    test_trues_inv = inverse_scale_preds(test_trues_s, scalers)
    mae, rmse, r2 = compute_metrics(test_preds_inv, test_trues_inv)
    print(f"\n>>> {name} TEST METRICS (real units): MAE={mae:.4f} RMSE={rmse:.4f} R2={r2:.4f}")

    results[name] = {
        "model": model,
        "mae": mae, "rmse": rmse, "r2": r2,
        "preds_scaled": test_preds_s, "trues_scaled": test_trues_s,
        "preds_real": test_preds_inv, "trues_real": test_trues_inv
    }

print("\nALL DONE. Results summary:")
for k,v in results.items():
    print(k, "-> MAE:", v['mae'], "RMSE:", v['rmse'], "R2:", v['r2'])


=== TRAINING LSTM ===
LSTM Epoch 1/30 | TrainLoss: 0.003793 | ValLoss: 0.001876 | time 39.7s
LSTM Epoch 2/30 | TrainLoss: 0.001599 | ValLoss: 0.001458 | time 21.5s
LSTM Epoch 4/30 | TrainLoss: 0.001252 | ValLoss: 0.001315 | time 4.8s
LSTM Epoch 6/30 | TrainLoss: 0.001184 | ValLoss: 0.001352 | time 4.2s
LSTM Epoch 8/30 | TrainLoss: 0.001177 | ValLoss: 0.001321 | time 30.6s
LSTM Epoch 10/30 | TrainLoss: 0.001135 | ValLoss: 0.001417 | time 33.2s
LSTM Epoch 12/30 | TrainLoss: 0.001122 | ValLoss: 0.001374 | time 31.7s
LSTM Epoch 14/30 | TrainLoss: 0.001095 | ValLoss: 0.001338 | time 34.9s
LSTM Epoch 16/30 | TrainLoss: 0.001083 | ValLoss: 0.001335 | time 32.7s
LSTM Epoch 18/30 | TrainLoss: 0.001067 | ValLoss: 0.001333 | time 31.9s
Early stopping triggered.

>>> LSTM TEST METRICS (real units): MAE=40.9622 RMSE=99.4018 R2=0.8452

=== TRAINING CNNLSTM ===
CNNLSTM Epoch 1/30 | TrainLoss: 0.003625 | ValLoss: 0.001858 | time 27.3s
CNNLSTM Epoch 2/30 | TrainLoss: 0.001526 | ValLoss: 0.001549 | tim